# 🚀 GLM-OCR: Open-Source Scientific PDF-to-Markdown Benchmark on Google Colab
### With Persistent Google Drive Model Caching (Never Re-Download Weights!)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ri5hiPrajapati/pdf-to-markdown-verbatim/blob/main/colab/glm_ocr_benchmark.ipynb)

This notebook runs **Zhipu AI's GLM-OCR** (`zai-org/GLM-OCR`) on Google Colab's free **Tesla T4 GPU (15.3 GB VRAM)**.

### Key Highlights:
* **Google Drive Persistence**: Model weights (~2.5 GB) are saved directly to your Google Drive. After downloading once, subsequent sessions load the model in **3–5 seconds**.
* **Model Architecture**: CogViT visual encoder (~0.4B) + GLM-0.5B language decoder = **~0.9B parameters** (~2.5 GB VRAM in BF16/FP16).
* **Domain Accuracy**: Specialized for dense scientific layouts, mathematical LaTeX equations, and multi-row tables.

In [ ]:
# Step 1: Verify Hardware Accelerator
!nvidia-smi

In [ ]:
# Step 2: Install Dependencies
!pip install -q transformers>=4.45.0 torch torchvision pymupdf pillow accelerate markdown

In [ ]:
# Step 3: Mount Google Drive & Configure Persistent Hugging Face Cache
# This ensures model weights (~2.5 GB) are stored permanently in your Drive
# and NEVER need to be re-downloaded when your Colab GPU reconnects!
import os
from google.colab import drive, userdata

print("1. Mounting Google Drive...")
drive.mount('/content/drive')

# Configure persistent cache path in Google Drive
hf_cache_dir = '/content/drive/MyDrive/huggingface_cache'
os.makedirs(hf_cache_dir, exist_ok=True)
os.environ['HF_HOME'] = hf_cache_dir
os.environ['TRANSFORMERS_CACHE'] = hf_cache_dir

# Automatically inject HF_TOKEN if saved in Colab Secrets (silences unauthenticated warnings)
try:
    token = userdata.get('HF_TOKEN')
    if token:
        os.environ['HF_TOKEN'] = token.strip()
        print("2. ✅ HF_TOKEN authenticated successfully from Colab Secrets!")
    else:
        print("2. ℹ️ HF_TOKEN not found in Secrets. Using standard public download.")
except Exception:
    print("2. ℹ️ Secrets manager skipped.")

print(f"3. ✅ Persistent Model Cache Active: {hf_cache_dir}")
print("   Weights will download directly to your Google Drive and persist indefinitely!")

In [ ]:
# Step 4: Clone Benchmark Toolkit & Verification Harness
!git clone https://github.com/Ri5hiPrajapati/pdf-to-markdown-verbatim.git
%cd pdf-to-markdown-verbatim

In [ ]:
# Step 5: Load zai-org/GLM-OCR into GPU Memory
# (Loads from Google Drive if already downloaded; otherwise downloads and saves to Drive)
import torch
from transformers import AutoProcessor, AutoModelForCausalLM

model_id = "zai-org/GLM-OCR"
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading {model_id} on {device} using persistent cache in Google Drive...")

processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    trust_remote_code=True
)

print("✅ GLM-OCR successfully loaded into VRAM!")

In [ ]:
# Step 6: Define Verbatim PDF Parsing Pipeline
import io
import time
import pymupdf as fitz
from PIL import Image

def parse_pdf_with_glm(pdf_path, dpi=150):
    doc = fitz.open(pdf_path)
    print(f"Processing: {pdf_path} ({len(doc)} pages)")
    
    page_mds = []
    t0 = time.time()
    
    for idx, page in enumerate(doc):
        pt0 = time.time()
        pix = page.get_pixmap(dpi=dpi)
        image = Image.open(io.BytesIO(pix.tobytes("png"))).convert("RGB")
        
        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image},
                    {"type": "text", "text": "Text Recognition: Extract all text, headings, equations, and tables verbatim into Markdown."}
                ]
            }
        ]
        
        inputs = processor.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_dict=True,
            return_tensors="pt"
        ).to(model.device)
        
        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=4096)
            
        prompt_len = inputs["input_ids"].shape[-1]
        generated_text = processor.decode(outputs[0][prompt_len:], skip_special_tokens=True).strip()
        page_mds.append(generated_text)
        print(f"  • Page {idx+1}/{len(doc)}: done in {time.time()-pt0:.2f}s ({len(generated_text)} chars)")
        
    total_time = time.time() - t0
    full_markdown = "\n\n---\n\n".join(page_mds)
    out_path = pdf_path.replace(".pdf", "_glm_ocr.md")
    with open(out_path, "w", encoding="utf-8") as f:
        f.write(full_markdown)
        
    print(f"\n🎉 Conversion complete in {total_time:.2f}s ({total_time/len(doc):.2f}s/page)!")
    print(f"Saved to: {out_path}")
    return out_path

In [ ]:
# Step 7: Upload a Benchmark PDF (e.g. Research Article 4)
from google.colab import files
import shutil

print("Upload your benchmark PDF file (or cancel to use default demo):")
uploaded = files.upload()

if uploaded:
    target_pdf = list(uploaded.keys())[0]
else:
    target_pdf = "sample_paper.pdf"
    !wget -q -O sample_paper.pdf "https://arxiv.org/pdf/2401.00001.pdf"

converted_md = parse_pdf_with_glm(target_pdf)

# Automatically save a permanent backup copy in Google Drive
drive_out_dir = '/content/drive/MyDrive/pdf_markdown_outputs'
os.makedirs(drive_out_dir, exist_ok=True)
shutil.copy(converted_md, os.path.join(drive_out_dir, os.path.basename(converted_md)))
print(f"✅ Output safely preserved in Google Drive: {drive_out_dir}/{os.path.basename(converted_md)}")

In [ ]:
# Step 8: Run Quantitative Audit via verify_md.py
!python verify_md.py "{target_pdf}" "{converted_md}"

In [ ]:
# Step 9: Download Converted Markdown to Local Machine
files.download(converted_md)